###     3(b) Write a PyTorch program to classify IMDb or Amazon reviews using an LSTM. 

In [34]:
!pip install pandas numpy scikit-learn torch tensorflow



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [36]:
data = pd.read_csv("IMDB Dataset.csv")
data['sentiment'] = data['sentiment'].map({'positive': 1, 'negative': 0})

train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data['review'])

X_train = tokenizer.texts_to_sequences(train_data['review'])
X_test = tokenizer.texts_to_sequences(test_data['review'])

X_train = pad_sequences(X_train, maxlen=200)
X_test = pad_sequences(X_test, maxlen=200)

y_train = train_data['sentiment'].values
y_test = test_data['sentiment'].values

X_train_tensor = torch.tensor(X_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)


In [37]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)


In [38]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        out = self.fc(h_n[-1])
        return self.sigmoid(out)


The below epochs will take time, dont hurry, can take min of 15-20 mins to run

In [39]:
model = SentimentLSTM(vocab_size=5000, embedding_dim=128, hidden_dim=128)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    for inputs, labels in train_loader:
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.4635
Epoch 2, Loss: 0.3225
Epoch 3, Loss: 0.2485
Epoch 4, Loss: 0.3728
Epoch 5, Loss: 0.3738


In [40]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs).squeeze()
        predicted = (outputs >= 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 87.78%


In [41]:
# Assuming tokenizer from earlier
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [44]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def predict_sentiment(review):
    model.eval()
    sequence = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequence, maxlen=200)
    input_tensor = torch.tensor(padded_sequence, dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(input_tensor).squeeze()
        return "positive" if output.item() > 0.5 else "negative"


In [45]:
print(predict_sentiment("This movie was fantastic!"))
print(predict_sentiment("The movie was boring and too long."))
print(predict_sentiment("It was okay, not great but not bad either."))


positive
negative
negative
